## Extending the CI/CD Pipeline and Kubernetes Deployment you built
> This notebook is a **companion to `week6.ipynb`**. It does not repeat the theory but it gives you small, guided modifications to the files you already worked with.

### Before you start
- Make sure you completed `week6.ipynb` first. You'll be editing files you already opened there: `.github/workflows/ci.yml`, `k8s/deployment.yaml`, and `k8s/service.yaml`.
- Keep the **Week6 directory** open in your terminal/editor, same as before.
- Each exercise below tells you **which file to open**, **exactly what to add**, and **what output you should see** when it works. If your output doesn't match, that's the bug to fix. Don't move on until it does.

### How this notebook is structured?
| Exercise | File you edit | Concept practiced |
|---|---|---|
| 1 | `.github/workflows/ci.yml` | Adding a lint stage to a CI pipeline |
| 2 | `k8s/deployment.yaml` | Scaling replicas + wiring in a ConfigMap |
| 3 | `k8s/service.yaml` | Changing a Service type and understanding the trade-off |

## **Problem 1: Add a lint stage to the CI pipeline**

> **File to open:** [`.github/workflows/ci.yml`](.github/workflows/ci.yml)

> Right now the workflow only runs `pytest`. Your task is to add a second job that runs a linter (`flake8`) so style problems get caught automatically too, just like test failures.

**Steps:**
1. Add `flake8==7.0.0` as a new line in [`requirements.txt`](requirements.txt).
2. Open [`.github/workflows/ci.yml`](.github/workflows/ci.yml).
3. Add a **second job** below `build-and-test`, at the same indentation level (i.e. still under `jobs:`):

```yaml
  lint:
    runs-on: ubuntu-latest
    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run flake8
        run: flake8 app.py test_app.py --max-line-length=100
```

4. Save the file. If you have this pushed to GitHub (from `week6.ipynb`), commit and push &rarr; you should now see **two** jobs (`build-and-test` and `lint`) run side by side in the Actions tab.

**Expected output** (running locally first, to check before pushing):
```bash
pip install -r requirements.txt
flake8 app.py test_app.py --max-line-length=100
```
> This should print **no output** and exit with code 0 if `app.py`/`test_app.py` are clean. Run the cell below to check locally.


In [ ]:
import subprocess

result = subprocess.run(
    ["flake8", "app.py", "test_app.py", "--max-line-length=100"],
    cwd=".", capture_output=True, text=True
)
print("STDOUT:", result.stdout or "(clean - no issues found)")
print("STDERR:", result.stderr)
print("Exit code:", result.returncode)


**Question:** Why does it make sense for `lint` to be a **separate job** from `build-and-test`, rather than an extra step added inside the same job? (Hint: what happens to the other job if one job fails, and does that matter here?)

## **Problem 2: Scale replicas and wire in a ConfigMap**

**File to open:** [`k8s/deployment.yaml`](k8s/deployment.yaml)

> Right now `APP_VERSION` is hardcoded directly in `deployment.yaml`, and there are only 2 replicas. Your task is to (a) scale to 3 replicas, and (b) read `APP_VERSION` from the ConfigMap in [`k8s/configmap.yaml`](k8s/configmap.yaml) instead of hardcoding it.

**Steps:**
1. Open [`k8s/configmap.yaml`](k8s/configmap.yaml) &rarr; note it already defines `APP_VERSION: "1.0.0"` under `data:`.
2. Open [`k8s/deployment.yaml`](k8s/deployment.yaml). Change `replicas: 2` to `replicas: 3`.
3. Replace the existing `env:` block:
```yaml
          env:
            - name: APP_VERSION
              value: "1.0.0"
```
   with a version that pulls the value from the ConfigMap instead:
```yaml
          env:
            - name: APP_VERSION
              valueFrom:
                configMapKeyRef:
                  name: flask-config
                  key: APP_VERSION
```
4. Apply the ConfigMap **before** the Deployment (Kubernetes needs the ConfigMap to exist first):
```bash
kubectl apply -f k8s/configmap.yaml
kubectl apply -f k8s/deployment.yaml
```
5. Check the rollout:
```bash
kubectl get pods
kubectl get deployment flask-deployment
```

**Expected output:**
- `kubectl get pods` → **3** Pods with `app=flask-app`, all `Running`.
- Visiting the app's `/` route (via `minikube service flask-service --url`, from `week6.ipynb`) → still shows `{"message": ..., "version": "1.0.0"}`, but the version now comes from the ConfigMap, not a hardcoded string in the Deployment.

**Questions** [Don't forget to answer these reflection questions]
- If you now edit `k8s/configmap.yaml` to change `APP_VERSION` to `"2.0.0"` and re-run `kubectl apply -f k8s/configmap.yaml`, will already-running Pods pick up the new value immediately? What does that tell you about when environment variables from a ConfigMap are actually read?
- Why is separating configuration (ConfigMap) from the Deployment spec generally considered good practice, especially across `dev`/`staging`/`prod` namespaces?

## **Problem 3: Change the Service type**

**File to open:** [`k8s/service.yaml`](k8s/service.yaml)

> `flask-service` is currently a `NodePort`, which is convenient for local development but not how you'd expose a service to the public internet from a real cloud cluster. Your task is to see what changes when you switch it to `ClusterIP` (the default, internal-only type).

**Steps:**
1. Open [`k8s/service.yaml`](k8s/service.yaml).
2. Change:
```yaml
  type: NodePort
```
   to:
```yaml
  type: ClusterIP
```
   and remove the `nodePort: 30036` line entirely (ClusterIP Services don't take a static node port).

3. Re-apply:
```bash
kubectl apply -f k8s/service.yaml
kubectl get services
```
4. Try reaching it the old way:
```bash
minikube service flask-service --url
```

**Expected output:**
- `kubectl get services` → `flask-service` now shows `TYPE  ClusterIP` (no node port listed).
- `minikube service flask-service --url` → **fails or errors**, since `ClusterIP` Services are only reachable *from inside the cluster*, not from your host machine.
5. Confirm it's still reachable from **inside** the cluster by running a temporary Pod:
```bash
kubectl run tmp-curl --rm -it --image=curlimages/curl -- curl http://flask-service:5000/
```
> This should return the same JSON response as before &rarr; proving the Service still works, just not from outside the cluster.
6. Change `k8s/service.yaml` back to `NodePort` (with `nodePort: 30036`) and re-apply, so the rest of the lab keeps working:
```bash
kubectl apply -f k8s/service.yaml
```

**Question:** In a real cloud deployment, what Kubernetes Service `type` would you use to expose `flask-service` to the public internet with a real load balancer, and how does it differ from both `ClusterIP` and `NodePort`?

### Well done, Week 6 Exercises Complete 🎉!
- Don't forget to submit your [exercise notebook](week6_exercise.ipynb) on **Moodle**.

- See you next time!